# Caso de Negocio: Promise & Speed — Análisis en Python

**Autor:** David Adrián González Molina
**Ruta analizada:** Zacatecas → Oaxaca · **Año:** 2024

**Contexto.** La tasa de conversión de compra (CVR) del Marketplace cayó de 70.3% (2023) a
66.5% (2024). El equipo de Inteligencia Logística busca recuperarla mejorando el diseño de la
promesa de entrega en la ruta más ofensora (ZAC–OAX), minimizando entregas anticipadas y
tardías. Este notebook cubre la limpieza, la auditoría de calidad y el análisis de los KPIs de conversión y cumplimiento de la ruta.

> **Cómo correrlo:** coloca el archivo `CasoNegocioPromiseSpeed Jun_25.xlsx` en la misma
> carpeta que este notebook y ejecuta las celdas de arriba hacia abajo.

In [1]:
# Importación de librerías
import pandas as pd   # Manejo y análisis de datos tabulares

pd.set_option('display.width', 120)
print("Pandas cargado correctamente:", pd.__version__)

Pandas cargado correctamente: 2.3.3


## 1. Carga de datos
Leemos la hoja `Datos` del Excel original (ruta relativa para que corra en cualquier equipo).

In [2]:
# Ruta relativa: el archivo debe estar en la misma carpeta que el notebook
ARCHIVO = 'CasoNegocioPromiseSpeed Jun_25.xlsx'

df = pd.read_csv(ARCHIVO) if ARCHIVO.endswith('.csv') else pd.read_excel(ARCHIVO, sheet_name='Datos')
print(f"Registros cargados: {len(df):,}")
df.head()

Registros cargados: 36,544


,ANIO,MES,SEMANA,HORA_COMPRA,HORA_ENTREGA,ORIGEN,DESTINO,EJECUCION_ENTREGA,ENTREGA_DEMORADA,ENTREGA_ANTICIPADA,ENTREGA_A_TIEMPO,CANCELADO,TOTAL_ENVIOS,VISITAS
0,2024,3,9,11,12.0,ZACATECAS,OAXACA,ON_TIME,0,0,10,0,10,16
1,2024,3,9,21,15.0,ZACATECAS,OAXACA,DELAY,1,0,0,0,1,2
2,2024,3,9,22,14.0,ZACATECAS,OAXACA,EARLY,0,9,0,0,9,17
3,2024,3,9,9,20.0,ZACATECAS,OAXACA,EARLY,0,4,0,0,4,5
4,2024,3,9,20,16.0,ZACATECAS,OAXACA,EARLY,0,5,0,0,5,5


## 2. Limpieza y filtrado (Data Governance)
Estandarizamos texto y aislamos la ruta estratégica del año de interés. Usamos `.copy()` para
trabajar sobre un DataFrame independiente y evitar el `SettingWithCopyWarning`.

In [3]:
# Estandarización de texto: sin espacios accidentales y en MAYÚSCULAS
df['ORIGEN']  = df['ORIGEN'].str.strip().str.upper()
df['DESTINO'] = df['DESTINO'].str.strip().str.upper()

# Filtrado estratégico: 2024 + ruta Zacatecas -> Oaxaca
df_clean = df[
    (df['ANIO'] == 2024) &
    (df['ORIGEN'] == 'ZACATECAS') &
    (df['DESTINO'] == 'OAXACA')
].copy()

print(f"Registros tras el filtro: {len(df_clean):,}")
print(f"Origen único:  {df_clean['ORIGEN'].unique()}")
print(f"Destino único: {df_clean['DESTINO'].unique()}")

Registros tras el filtro: 36,544
Origen único:  ['ZACATECAS']
Destino único: ['OAXACA']


## 3. Auditoría de calidad: nulos, duplicados y anomalías
Antes de analizar, validamos la integridad de los datos. Los nulos en `HORA_ENTREGA` son
esperables en compras canceladas (nunca se entregaron); lo que sí es una **anomalía** es un
paquete sin hora de entrega que NO está cancelado.

In [4]:
# Duplicados exactos
print(f"Duplicados exactos: {df_clean.duplicated().sum()}")

# Nulos por columna (solo las que tienen al menos uno)
nulos = df_clean.isnull().sum()
print("\nNulos por columna:")
print(nulos[nulos > 0])

# ¿Qué estatus tienen los registros sin hora de entrega?
sin_hora = df_clean[df_clean['HORA_ENTREGA'].isnull()]
print("\nEstatus de los registros sin HORA_ENTREGA:")
print(sin_hora['EJECUCION_ENTREGA'].value_counts())

Duplicados exactos: 0

Nulos por columna:
HORA_ENTREGA    1128
dtype: int64

Estatus de los registros sin HORA_ENTREGA:
EJECUCION_ENTREGA
CANCELLED    1113
DELAY          13
EARLY           1
ON_TIME         1
Name: count, dtype: int64


In [5]:
# Aislamos las ANOMALÍAS: sin hora de entrega pero NO canceladas
anomalias = df_clean[
    (df_clean['HORA_ENTREGA'].isnull()) &
    (df_clean['EJECUCION_ENTREGA'] != 'CANCELLED')
].copy()

anomalias.to_csv('evidencia_anomalias_sistema.csv', index=False)
print(f"Registros anómalos aislados: {len(anomalias)}")
anomalias[['MES', 'HORA_COMPRA', 'EJECUCION_ENTREGA', 'TOTAL_ENVIOS', 'VISITAS']]

Registros anómalos aislados: 15


,MES,HORA_COMPRA,EJECUCION_ENTREGA,TOTAL_ENVIOS,VISITAS
2277,3,13,EARLY,1,2
2429,3,15,DELAY,1,1
4468,4,10,DELAY,1,1
10170,5,14,DELAY,1,1
17505,7,12,DELAY,1,2
22760,9,23,DELAY,1,2
25194,9,11,DELAY,1,1
28817,10,13,DELAY,1,1
30792,11,7,DELAY,1,1
31427,11,13,DELAY,1,2


## Análisis exploratorio de los datos
Primeras filas y resumen estadístico de las columnas numéricas.

In [6]:
print("Primeras 5 filas:")
display(df_clean.head())

print("\nResumen estadístico:")
df_clean.describe()

Primeras 5 filas:


,ANIO,MES,SEMANA,HORA_COMPRA,HORA_ENTREGA,ORIGEN,DESTINO,EJECUCION_ENTREGA,ENTREGA_DEMORADA,ENTREGA_ANTICIPADA,ENTREGA_A_TIEMPO,CANCELADO,TOTAL_ENVIOS,VISITAS
0,2024,3,9,11,12.0,ZACATECAS,OAXACA,ON_TIME,0,0,10,0,10,16
1,2024,3,9,21,15.0,ZACATECAS,OAXACA,DELAY,1,0,0,0,1,2
2,2024,3,9,22,14.0,ZACATECAS,OAXACA,EARLY,0,9,0,0,9,17
3,2024,3,9,9,20.0,ZACATECAS,OAXACA,EARLY,0,4,0,0,4,5
4,2024,3,9,20,16.0,ZACATECAS,OAXACA,EARLY,0,5,0,0,5,5



Resumen estadístico:


,ANIO,MES,SEMANA,HORA_COMPRA,HORA_ENTREGA,ENTREGA_DEMORADA,ENTREGA_ANTICIPADA,ENTREGA_A_TIEMPO,CANCELADO,TOTAL_ENVIOS,VISITAS
count,36544.0,36544.000000,36544.000000,36544.000000,35416.000000,36544.000000,36544.000000,36544.000000,36544.000000,36544.000000,36544.000000
mean,2024.0,7.437473,30.789021,12.931151,15.118365,0.347034,4.557191,4.016911,0.303525,9.224661,13.863808
std,0.0,2.882568,12.661211,6.493435,4.081887,1.057276,9.617301,8.530699,2.219635,11.105421,17.206842
min,2024.0,3.000000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,2024.0,5.000000,20.000000,8.000000,12.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000
50%,2024.0,7.000000,31.000000,13.000000,15.000000,0.000000,0.000000,0.000000,0.000000,4.000000,6.000000
75%,2024.0,10.000000,42.000000,18.000000,18.000000,0.000000,4.000000,3.000000,0.000000,13.000000,20.000000
max,2024.0,12.000000,53.000000,23.000000,23.000000,27.000000,131.000000,97.000000,78.000000,131.000000,188.000000


## Conversión (CVR) por hora de compra
`CVR = TOTAL_ENVIOS / VISITAS`, agregado por hora y ordenado de mayor a menor.

In [7]:
cvr_hora = (df_clean.groupby('HORA_COMPRA')
            .agg(TOTAL_ENVIOS=('TOTAL_ENVIOS', 'sum'),
                 VISITAS=('VISITAS', 'sum'))
            .reset_index())

cvr_hora['CVR'] = cvr_hora['TOTAL_ENVIOS'] / cvr_hora['VISITAS']
cvr_hora_sorted = cvr_hora.sort_values('CVR', ascending=False)

print("Top 5 horas con mejor tasa de conversión (CVR):")
cvr_hora_sorted.head()

Top 5 horas con mejor tasa de conversión (CVR):


,HORA_COMPRA,TOTAL_ENVIOS,VISITAS,CVR
23,23,10849,16067,0.675235
3,3,1056,1570,0.672611
12,12,21622,32150,0.672535
5,5,1763,2628,0.670852
14,14,20072,29928,0.670676


## Cumplimiento de entrega (on-time) por mes
Proporción de envíos entregados a tiempo respecto al total mensual.

In [8]:
otp_mensual = (df_clean.groupby('MES')
               .agg(ENTREGA_A_TIEMPO=('ENTREGA_A_TIEMPO', 'sum'),
                    TOTAL_ENVIOS=('TOTAL_ENVIOS', 'sum'))
               .reset_index())

otp_mensual['PCT_ON_TIME'] = (otp_mensual['ENTREGA_A_TIEMPO'] / otp_mensual['TOTAL_ENVIOS']) * 100

print("Porcentaje de entregas a tiempo por mes:")
otp_mensual[['MES', 'PCT_ON_TIME']].round(1)

Porcentaje de entregas a tiempo por mes:


,MES,PCT_ON_TIME
0,3,37.0
1,4,45.7
2,5,36.7
3,6,48.0
4,7,52.1
5,8,48.5
6,9,44.9
7,10,45.0
8,11,39.7
9,12,38.8


## Hora óptima de compra: mejor CVR con cero entregas demoradas
Aislamos las operaciones sin entregas demoradas y recalculamos el CVR para encontrar la
"hora de oro": alta conversión sin comprometer el cumplimiento.

In [9]:
df_sin_demoras = df_clean[df_clean['ENTREGA_DEMORADA'] == 0]

cvr_sd = (df_sin_demoras.groupby('HORA_COMPRA')
          .agg(TOTAL_ENVIOS=('TOTAL_ENVIOS', 'sum'),
               VISITAS=('VISITAS', 'sum'))
          .reset_index())
cvr_sd['CVR'] = cvr_sd['TOTAL_ENVIOS'] / cvr_sd['VISITAS']
cvr_sd = cvr_sd.sort_values('CVR', ascending=False)

print("Top 3 horas con mejor CVR y sin demoras:")
display(cvr_sd.head(3))

mejor = cvr_sd.iloc[0]
print(f"\nConclusión: la hora de compra con mejor CVR sin demoras es las "
      f"{int(mejor['HORA_COMPRA'])}:00 hrs, con un CVR de {mejor['CVR']*100:.2f}%.")

Top 3 horas con mejor CVR y sin demoras:


,HORA_COMPRA,TOTAL_ENVIOS,VISITAS,CVR
23,23,10160,15074,0.674008
12,12,20757,30849,0.672858
3,3,1017,1512,0.672619



Conclusión: la hora de compra con mejor CVR sin demoras es las 23:00 hrs, con un CVR de 67.40%.


## 4. Exportación de la base auditada
Guardamos el dataset limpio para alimentar el dashboard de Tableau.

In [10]:
SALIDA = 'base_limpia_zac_oax_2024.csv'
df_clean.to_csv(SALIDA, index=False)
print(f"Base exportada: '{SALIDA}'  ({len(df_clean):,} registros)")

Base exportada: 'base_limpia_zac_oax_2024.csv'  (36,544 registros)
